In [16]:
# ============================================================
# CARE DASHBOARD — MASTER NOTEBOOK
# Climate Awareness and Risk Evaluation Dashboard
# ============================================================
# Student:    Ritesh Raju Ghorpade (202559288)
# Programme:  MSc Advanced Computer Science with Data Science
# University: University of Strathclyde, Glasgow
# Supervisor: Dr Daniel Thomas
# Deadline:   17th August 2026, noon UK time
# ============================================================
#
# NOTEBOOK STRUCTURE:
# ─────────────────────────────────────────────────────────────
# PART 1 — DATA COLLECTION
#   1.1  Library imports and path configuration
#   1.2  SEPA flood zones
#   1.3  OpenStreetMap data
#   1.4  NASA SRTM elevation
#   1.5  Met Office HadUK-Grid rainfall
#
# PART 2 — EXPLORATORY DATA ANALYSIS
#   2.1  SEPA PVA analysis
#   2.2  OSM buildings analysis
#   2.3  OSM roads analysis
#   2.4  OSM water bodies analysis
#   2.5  Elevation analysis
#   2.6  Rainfall analysis
#
# PART 3 — FEATURE ENGINEERING
#   3.1  Create 100m study grid
#   3.2  Attach elevation
#   3.3  Calculate distance to water
#   3.4  Calculate building density
#   3.5  Calculate road density
#   3.6  Attach rainfall features
#   3.7  Create flood risk label
#   3.8  Correlation analysis
#   3.9  Export feature matrix
#
# PART 4 — MACHINE LEARNING
#   4.1  Load and prepare data
#   4.2  Train/test split
#   4.3  Baseline Random Forest
#   4.4  Hyperparameter tuning
#   4.5  Model evaluation
#   4.6  SHAP analysis
#
# PART 5 — RESULTS AND VISUALISATION
#   5.1  Feature importance
#   5.2  Risk map
#   5.3  SHAP summary plots
# ============================================================

print("=" * 60)
print("CARE Dashboard — Master Notebook")
print("Climate Awareness and Risk Evaluation Dashboard")
print("Ritesh Raju Ghorpade | 202559288")
print("University of Strathclyde | August 2026")
print("=" * 60)

CARE Dashboard — Master Notebook
Climate Awareness and Risk Evaluation Dashboard
Ritesh Raju Ghorpade | 202559288
University of Strathclyde | August 2026


In [17]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

Libraries loaded!


In [18]:
import os

BASE = "/Users/riteshghorpade/Documents/010_Project/002_Dataset"

files = {
    "SEPA PVA":          f"{BASE}/001_SEPA/GeoPackage/Data/PVAv2.gpkg",
    "OSM Buildings":     f"{BASE}/002_OSM/osm_buildings_glasgow.gpkg",
    "OSM Roads":         f"{BASE}/002_OSM/osm_roads_glasgow.gpkg",
    "OSM Water":         f"{BASE}/002_OSM/osm_water_glasgow.gpkg",
    "NASA Elevation":    f"{BASE}/003_NASA/nasa_elevation_glasgow.gpkg",
    "NASA SRTM Raw":     f"{BASE}/003_NASA/nasa_srtm_glasgow.tif",
    "HadUK Daily":       f"{BASE}/003_NASA/hadukgrid_daily_glasgow_clean.csv",
    "HadUK Derived":     f"{BASE}/003_NASA/hadukgrid_derived_features.csv",
    "Feature Matrix":    f"{BASE}/feature_matrix.csv",
    "HadUK Monthly":     f"{BASE}/001_SEPA/scotland_rainfall_hadukgrid.csv",
}

print("Dataset availability check:")
print("=" * 50)
for name, path in files.items():
    exists  = os.path.exists(path)
    status  = "OK" if exists else "MISSING"
    if exists:
        size = os.path.getsize(path) / (1024*1024)
        print(f"  {name:<20} {status}  ({size:.1f} MB)")
    else:
        print(f"  {name:<20} {status}")
print("=" * 50)

Dataset availability check:
  SEPA PVA             OK  (10.7 MB)
  OSM Buildings        OK  (14.4 MB)
  OSM Roads            OK  (8.2 MB)
  OSM Water            OK  (0.3 MB)
  NASA Elevation       OK  (29.9 MB)
  NASA SRTM Raw        OK  (0.1 MB)
  HadUK Daily          OK  (0.5 MB)
  HadUK Derived        OK  (0.0 MB)
  Feature Matrix       OK  (0.6 MB)
  HadUK Monthly        OK  (0.0 MB)


In [19]:
BASE = "/Users/riteshghorpade/Documents/010_Project/002_Dataset"

SEPA_PATH    = f"{BASE}/001_SEPA/GeoPackage/Data/PVAv2.gpkg"
OSM_BUILD    = f"{BASE}/002_OSM/osm_buildings_glasgow.gpkg"
OSM_ROADS    = f"{BASE}/002_OSM/osm_roads_glasgow.gpkg"
OSM_WATER    = f"{BASE}/002_OSM/osm_water_glasgow.gpkg"
NASA_ELEV    = f"{BASE}/003_NASA/nasa_elevation_glasgow.gpkg"
RAIN_DAILY   = f"{BASE}/003_NASA/hadukgrid_daily_glasgow_clean.csv"
RAIN_DERIVED = f"{BASE}/003_NASA/hadukgrid_derived_features.csv"
MAPS_PATH    = f"{BASE}/004_Maps"

print("Paths defined!")

Paths defined!


In [20]:
# Read SEPA raw and view
import geopandas as gpd
df_sepa = gpd.read_file(SEPA_PATH)
df_sepa.head(10)

,OBJECTID_1,PVA_Name,Desig_Date,Chng_C1,PVA_Ref,Shape_Length,Shape_Area,geometry
0,1,Thurso and Halkirk,2018-12-18,Boundary change,02/01/01,84478.694740,1.027316e+08,"MULTIPOLYGON (((308974.999 970125, 308974.999 ..."
1,2,Wick,2018-12-18,Boundary change,02/01/02,100485.288215,8.650975e+07,"MULTIPOLYGON (((334273.843 952200.314, 334261...."
2,3,Lochinver,2018-12-18,Boundary change,02/01/03,57670.830257,4.344988e+07,"MULTIPOLYGON (((209625 923075, 209625 923025, ..."
3,4,Golspie,2011-12-22,No change,02/01/04,36035.606949,2.416975e+07,"MULTIPOLYGON (((283968.105 900005.925, 283962 ..."
4,5,Dornoch,2011-12-22,No change,02/01/05,58566.815141,4.052294e+07,"MULTIPOLYGON (((274774.999 896075, 274775 8960..."
5,6,Aird Point,2018-12-18,New PVA,02/01/06,52469.850623,5.206283e+07,"MULTIPOLYGON (((188555.999 897337.001, 188611 ..."
6,7,Gairloch,2018-12-18,New PVA,02/01/07,96088.344946,7.179356e+07,"MULTIPOLYGON (((180990 875134.999, 181025 8751..."
7,8,Tarbat Ness,2011-12-22,No change,02/01/08,116236.905978,7.762992e+07,"MULTIPOLYGON (((287371 876719, 287339.999 8766..."
8,9,Invergordon,2018-12-18,Boundary change,02/01/09,30046.544683,2.737568e+07,"MULTIPOLYGON (((270875 874925, 270875 874874.9..."
9,10,Alness,2011-12-22,No change,02/01/10,56861.146822,6.002366e+07,"MULTIPOLYGON (((260275 879825.001, 260275 8797..."


In [21]:
# Read OSM Buildings fresh from file
buildings_raw = gpd.read_file(OSM_BUILD)
buildings_raw.head(10)

,element,id,building,building:levels,height,name,geometry
0,node,1495513134,yes,None,None,The Hengler's Circus,POINT (258251.041 665933.24)
1,node,1705790482,yes,None,None,None,POINT (261171.756 663284.757)
2,node,1936303000,hotel,None,None,Fraser Suites,POINT (259568.006 664953.44)
3,node,2180409219,construction,None,None,Next,POINT (259150.854 665012.542)
4,node,2371954470,dormitory,None,None,Queen Margaret Halls,POINT (256448.68 668027.705)
5,node,2452854341,garage,None,None,None,POINT (261017.459 664194.037)
6,node,3218826826,university,None,None,Department of Theology & Religious Studies,POINT (256796.717 666714.266)
7,node,3813079919,commercial,None,None,Esso - Kelvinside,POINT (257520.079 667893.428)
8,node,5001302993,retail,None,None,Holland & Barrett,POINT (258752.96 665252.981)
9,node,5159005096,warehouse,None,None,Mammoet Ferry Transport (UK) BV,POINT (264035.06 661463.622)


In [22]:
# Read OSM Roads fresh from file
roads_raw = gpd.read_file(OSM_ROADS)
roads_raw.head(20)

,element,id,highway,name,geometry
0,node,194815,motorway_junction,Bothwell Street Interchange,POINT (257999.408 665090.116)
1,node,194816,motorway_junction,St George's Cross,POINT (257950.826 666001.954)
2,node,194825,motorway_junction,Townhead Interchange,POINT (260023.156 666227.352)
3,node,194829,motorway_junction,Blochairn,POINT (261052.051 665913.363)
4,node,352856,traffic_signals,None,POINT (264127.527 662433.314)
5,node,352986,motorway_junction,Cumbernauld Road,POINT (263118.393 666681.832)
6,node,352998,motorway_junction,Cumbernauld Road,POINT (263745.392 666389.767)
7,node,352999,motorway_junction,Provan,POINT (261910.505 666100.818)
8,node,353009,motorway_junction,Townhead Interchange,POINT (260495.526 665920.359)
9,node,353017,motorway_junction,St George's Cross,POINT (258487.404 666374.306)


In [23]:
# Read OSM Water fresh from file
water_raw = gpd.read_file(OSM_WATER)
water_raw.head(20)

,element,id,name,geometry
0,relation,919880,Nature Pond,"POLYGON ((257674.475 662316.729, 257677.239 66..."
1,relation,1230156,Hogganfield Loch,"POLYGON ((263898.758 667143.583, 263892.042 66..."
2,relation,1235452,None,"POLYGON ((261119.752 668741.567, 261131.394 66..."
3,relation,1235453,None,"POLYGON ((260913.581 668768.335, 260924.731 66..."
4,relation,1288901,River Clyde,"POLYGON ((268768.649 659687.368, 268781.771 65..."
5,relation,1630110,None,"POLYGON ((261960.785 665669.829, 261968.354 66..."
6,relation,1947036,None,"POLYGON ((262788.359 668120.029, 262802.511 66..."
7,relation,2099637,None,"POLYGON ((260345.266 663042.392, 260344.344 66..."
8,relation,2266953,River Clyde,"POLYGON ((255631.857 665944.366, 255634.096 66..."
9,relation,3118337,None,"POLYGON ((256554.22 662984.379, 256564.851 662..."


In [24]:
# Read NASA Elevation fresh from file
elevation_raw = gpd.read_file(NASA_ELEV)
elevation_raw.head(20)

,latitude,longitude,elevation,geometry
0,55.92,-4.350000,79,POINT (253242.407 672152.61)
1,55.92,-4.349722,79,POINT (253259.762 672152.02)
2,55.92,-4.349444,79,POINT (253277.118 672151.43)
3,55.92,-4.349167,79,POINT (253294.473 672150.84)
4,55.92,-4.348889,78,POINT (253311.828 672150.251)
5,55.92,-4.348611,78,POINT (253329.183 672149.661)
6,55.92,-4.348333,77,POINT (253346.539 672149.071)
7,55.92,-4.348056,77,POINT (253363.894 672148.482)
8,55.92,-4.347778,78,POINT (253381.249 672147.892)
9,55.92,-4.347500,78,POINT (253398.605 672147.303)


In [25]:
# Read HadUK-Grid daily rainfall fresh from file
df_rain_raw = pd.read_csv(RAIN_DAILY)
df_rain_raw.tail(20)

,date,year,month,day,x_bng,y_bng,latitude,longitude,dist_from_uni_m,rainfall_mm
4364,2025-12-27,2025,12,27,257500.0,662500.0,55.834680,-4.275571,4290.841992,0.022420
4365,2025-12-27,2025,12,27,262500.0,662500.0,55.836130,-4.195793,2929.389868,0.031961
4366,2025-12-27,2025,12,27,257500.0,667500.0,55.879574,-4.278198,4283.844652,0.013790
4367,2025-12-27,2025,12,27,262500.0,667500.0,55.881027,-4.198328,2919.130864,0.022080
4368,2025-12-28,2025,12,28,257500.0,662500.0,55.834680,-4.275571,4290.841992,0.000000
4369,2025-12-28,2025,12,28,262500.0,662500.0,55.836130,-4.195793,2929.389868,0.000000
4370,2025-12-28,2025,12,28,257500.0,667500.0,55.879574,-4.278198,4283.844652,0.000000
4371,2025-12-28,2025,12,28,262500.0,667500.0,55.881027,-4.198328,2919.130864,0.000000
4372,2025-12-29,2025,12,29,257500.0,662500.0,55.834680,-4.275571,4290.841992,0.000000
4373,2025-12-29,2025,12,29,262500.0,662500.0,55.836130,-4.195793,2929.389868,0.000000


In [26]:
# Read derived features fresh from file
df_derived_raw = pd.read_csv(RAIN_DERIVED)
df_derived_raw

,x_bng,y_bng,latitude,longitude,dist_from_uni_m,mean_annual_mm_day,mean_winter_mm_day,wet_days_per_year,max_daily_mm
0,257500.0,662500.0,55.834680,-4.275571,4290.841992,2.8455,3.3141,163.7,37.886
1,257500.0,667500.0,55.879574,-4.278198,4283.844652,2.9601,3.4095,166.3,39.458
2,262500.0,662500.0,55.836130,-4.195793,2929.389868,2.6335,3.0400,156.0,37.096
3,262500.0,667500.0,55.881027,-4.198328,2919.130864,2.9116,3.2935,167.3,38.714
